In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import torch
import gc

In [124]:
# clearing GPU cache:
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [125]:
# force garbage collection:
gc.collect()
print('cache cleared and garbage collected!')

cache cleared and garbage collected!


In [5]:
proj_path = "/content/drive/MyDrive/llm_from_scratch/src"
data_path = "/content/drive/MyDrive/llm_from_scratch/datasets"

In [6]:
import os, sys
sys.path.append(proj_path)
sys.path.append(data_path)
os.chdir(proj_path)
print(os.getcwd())

/content/drive/MyDrive/llm_from_scratch/src


In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [8]:
from loading_weights import gpt as model
print("model loaded successfully.")

File already exists and is up-to-date: gpt2/774M/checkpoint
File already exists and is up-to-date: gpt2/774M/encoder.json
File already exists and is up-to-date: gpt2/774M/hparams.json
File already exists and is up-to-date: gpt2/774M/model.ckpt.data-00000-of-00001
File already exists and is up-to-date: gpt2/774M/model.ckpt.index
File already exists and is up-to-date: gpt2/774M/model.ckpt.meta
File already exists and is up-to-date: gpt2/774M/vocab.bpe
model loaded successfully.


In [9]:
gc.collect()
torch.cuda.empty_cache()

In [10]:
import tiktoken, json
from torch.utils.data import Dataset
from transformers import TrainingArguments, Trainer

In [11]:
class JsonInstructionDataset(Dataset):
    def __init__(self, json_data, max_length=512,):
        self.encoding = tiktoken.get_encoding('gpt2')
        self.json_data = json_data
        self.max_length = max_length

    def __len__(self):
        return len(self.json_data)

    def __getitem__(self, idx):
        item = self.json_data[idx]

        text = f"### Instruction:\n{item['instruction']}\n\n### Input:\n{item['input']}\n\n### Response:\n{item['output']}"

        # tokenizing with tiktoken
        tokens = self.encoding.encode(text)

        # truncating if necessary
        if len(tokens) > self.max_length:
            tokens = tokens[:self.max_length]

        input_ids = tokens

        # padding if necessary
        if len(input_ids) < self.max_length:
            padding_length = self.max_length - len(input_ids)
            padding = [self.encoding.eot_token] * padding_length
            input_ids = input_ids + padding

        return {
            'input_ids': torch.tensor(input_ids, dtype=torch.long),
            'labels': torch.tensor(input_ids, dtype=torch.long)
        }

In [12]:
import json

In [13]:
with open("/content/drive/MyDrive/llm_from_scratch/datasets/alpaca_gpt4_data.json", "r", encoding="utf-8") as f:
    json_data_gpt4 = json.load(f)

In [14]:
print(len(json_data_gpt4))

52002


In [15]:
json_gpt4_train, json_gpt4_val = json_data_gpt4[2500:5000], json_data_gpt4[5000:6500]
print(len(json_gpt4_train))
print(len(json_gpt4_val))

2500
1000


In [16]:
json_gpt4_train[-1]

{'instruction': 'Convert the given text into an alliteration.',
 'input': 'Partnership and perseverance',
 'output': 'Persistent partnership with perseverance.'}

In [17]:
train_dataset = JsonInstructionDataset(json_gpt4_train)
val_dataset = JsonInstructionDataset(json_gpt4_val)

In [18]:
train_dataset[0]

{'input_ids': tensor([21017, 46486,    25,   198, 23318,  1115,  9040,   329, 10589,  5448,
            13,   198,   198, 21017, 23412,    25,   628,   198, 21017, 18261,
            25,   198,    16,    13, 27574,   257, 12974,   290, 48102,  5496,
            25,  6889,  1654,   534, 13840,   389, 19889,   286,   257,  4996,
           286, 15921,   290, 13701,    11, 10904,  7532,    11,  2187, 21824,
            11,   290,  5448, 27997,    13,   770,  5419,   284,  2148,   534,
          1767,   351,   262,  6393, 20901,   284,  2163,   379,   663,  1266,
           290,   460,  1037,  2948, 10726, 10040,    13,   198,   198,    17,
            13,  1985,   496,   287,  3218,  3518,  3842,    25, 32900,   318,
          8780,   329, 10941,  1913, 11945,    11, 12749,    11,   290, 21134,
          1535,    13, 36223,   329,   379,  1551,  6640,  2431,   286, 10768,
         43294,  5517,   393,  5441,  2431,   286, 31543,  5517,  1123,  1285,
            13,   198,   198,    18,   

In [76]:
from transformers import PretrainedConfig

class GPTConfig(PretrainedConfig):
    def __init__(self, **kwargs):
        # config
        self.vocab_size = 50257
        self.context_length = 1024
        self.emb_dim = 1280
        self.n_heads = 20
        self.n_layers = 36
        self.drop_rate = 0.1
        self.qkv_bias = True
        super().__init__(**kwargs)

# attaching config to model
model.config = GPTConfig()

In [77]:
import types

def hf_forward(self, input_ids=None, labels=None, **kwargs):

    batch_size, seq_len = input_ids.shape
    tok_embeds = self.tok_emb(input_ids)
    pos_embeds = self.pos_emb(torch.arange(seq_len, device=input_ids.device))
    x = tok_embeds + pos_embeds
    x = self.drop_emb(x)
    x = self.trf_blocks(x)
    x = self.final_norm(x)
    logits = self.out_head(x)

    # calculating loss if labels are present
    if labels is not None:
        loss_fct = torch.nn.CrossEntropyLoss()
        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()
        loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
        return {'loss': loss, 'logits': logits}

    return {'logits': logits}

# applying the patch
model.forward = types.MethodType(hf_forward, model)
print("Fixed forward method")

Fixed forward method


In [78]:
def prepare_inputs_for_generation(self, input_ids, **kwargs):
    return {"input_ids": input_ids, **kwargs}

if not hasattr(model, 'prepare_inputs_for_generation'):
    model.prepare_inputs_for_generation = prepare_inputs_for_generation.__get__(model, type(model))

In [79]:
from peft import LoraConfig, get_peft_model
def setup_lora_model(model):
    # reducing model size
    model = model.half()
    # using attention layers from my model
    target_modules = [
        "W_query",
        "W_key",
        "W_value",
        "out_proj",
    ]

    # LoRA configuration
    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=target_modules,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )

    # applying LoRA
    model = get_peft_model(model, lora_config)

    return model

# applying LoRA to model
model = setup_lora_model(model)

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:196: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [138]:
import transformers

transformers.logging.set_verbosity_info()


training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/llm_from_scratch/text_generation_model/instruction_model",
    num_train_epochs=2,
    per_device_train_batch_size=3,
    per_device_eval_batch_size=3,
    gradient_accumulation_steps=11,
    learning_rate=3e-5,
    fp16=True,
    logging_steps=10,
    eval_strategy="steps",  # Fixed: changed from eval_strategy
    eval_steps=30,
    save_strategy="no",
    report_to="none",
    torch_compile=False,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    optim="adamw_torch",
    dataloader_pin_memory=False,
    dataloader_drop_last=True,
    remove_unused_columns=False,
    # for better visibility of progress bar
    logging_dir="./logs",
    disable_tqdm=False,  # to ensure progress bar is enabled
)



PyTorch: setting up devices


In [139]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

Using auto half precision backend


In [140]:
model.train()
print("done")

done


In [141]:
os.environ["WANDB_DISABLED"] = "true"
import gc
gc.collect()
torch.cuda.empty_cache

print("Started training...")
trainer.train()
print("Training completed!")

Started training...


***** Running training *****
  Num examples = 2,500
  Num Epochs = 2
  Instantaneous batch size per device = 3
  Total train batch size (w. parallel, distributed & accumulation) = 33
  Gradient Accumulation steps = 11
  Total optimization steps = 152
  Number of trainable parameters = 2,949,120


Step,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
gc.collect()
torch.cuda.empty_cache()

In [56]:
# # After instruction fine-tuning, merge LoRA into base model
# merged_model = model.merge_and_unload()

# # Save the MERGED weights (now base model has instruction knowledge)
# torch.save(merged_model.state_dict(), "/content/drive/MyDrive/llm_from_scratch/text_generation_model/instruction_model/instruction_tuned_model_weights.pth")

# print("✅ Saved unified model with instruction knowledge")

✅ Saved unified model with instruction knowledge


In [55]:
# def test_model(model, instruction, input_text="", max_new_tokens=1024):
#     # Format the prompt like during training
#     if input_text:
#         prompt = f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n"
#     else:
#         prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"

#     # Tokenize
#     encoding = tiktoken.get_encoding('gpt2')
#     input_ids = encoding.encode(prompt)
#     input_tensor = torch.tensor([input_ids]).to(device)  # Fixed: model.device instead of device

#     # Manual generation (no .generate() method)
#     model.eval()
#     with torch.no_grad():
#         generated = input_tensor

#         for i in range(max_new_tokens):
#             # Forward pass
#             outputs = model(input_ids=generated)
#             logits = outputs['logits']

#             # Get last token logits
#             next_token_logits = logits[0, -1, :]

#             # Apply temperature and sample
#             next_token_logits = next_token_logits / 0.7  # temperature
#             probs = torch.softmax(next_token_logits, dim=-1)
#             next_token = torch.multinomial(probs, num_samples=1)

#             # Stop if EOT token is generated BEFORE appending
#             if next_token.item() == encoding.eot_token:
#                 break

#             # Append to generated sequence
#             generated = torch.cat([generated, next_token.unsqueeze(0)], dim=1)

#     # Decode
#     response = encoding.decode(generated[0].tolist())
#     # Extract only the response part (after "### Response:\n")
#     response_text = response.split("### Response:\n")[-1]

#     # Remove endoftext token if it exists and clean up
#     response_text = response_text.replace('<|endoftext|>', '').strip()

#     return response_text

# # Test examples
# print("🧪 Testing the fine-tuned model:\n")

# # Example 1: General instruction
# test1 = test_model(
#     model,
#     instruction="write and essay on the topic of AI in 500 words.",
#     input_text="",
# )
# print(f"Test 1 - Explanation:\n{test1}\n")

🧪 Testing the fine-tuned model:

Test 1 - Explanation:
AI is a massive topic, so we will start with something simple, and then move on to more complex topics.

First, AI is a computer program that can perform a complex task such as understanding natural language, learning from experience, or interacting with other humans. We can think of AI as a machine that can learn from experience and make a better decision based on the experience.

In the past, AI programs were limited to performing repetitive tasks that were only useful for the purpose of achieving a goal. However, with advancements in technology, these tasks can now be automated to a level that is more useful for everyday tasks.

For example, the Google Brain project, which was started by Google in 2012, uses deep learning technology to teach computers to recognize objects in images and recognize spoken language, among other tasks.

Today, AI programs are capable of more complex tasks. For example, AI programs can learn to recogn